# Data Pipeline (Part I)

This notebook orchestrates the data pipeline in three stages:
1. Full fetch
2. Incremental update
3. Validation report

## Expected File Structure

```text
coinglass/
|-- data/
|   |-- *.csv                          # endpoint outputs (one file per endpoint+interval)
    |-- aligned/                       # you will see how it is constructed in this notebook
|   |-- _validation_report.json        # validation report (JSON)
|   `-- _validation_report.csv         # validation report (CSV)
|-- tool/
|   |-- fetch_all_endpoints.py         # full historical fetch + gap backfill
|   |-- update_all_endpoint.py         # incremental update (append new time_ms only)
|   `-- validate_downloaded_data.py    # data quality validation + optional cleanup
|-- main/
|   |-- Data_Pipeline.ipynb            # this orchestration notebook
|   `-- Coinglass_Connection_Check.ipynb
```

In [ ]:
from pathlib import Path
import subprocess
import sys

ROOT = Path.cwd()
if not (ROOT / "tool").exists() and (ROOT.parent / "tool").exists():
    ROOT = ROOT.parent

print("Project root:", ROOT)
print("Python:", sys.executable)

## 1) Full Fetch (only for first-time user)

Run the full historical collection pipeline (includes in-script dedupe and gap backfill).

In [2]:
# subprocess.run([sys.executable, str(ROOT / "tool" / "fetch_all_endpoints.py")], check=False)

## 2) Incremental Update

Only append new rows to existing CSVs in `data/` (no full overwrite).

In [ ]:
subprocess.run([sys.executable, str(ROOT / "tool" / "update_all_endpoint.py")], check=False)

## 3) Validate Data Quality

Generate validation reports:
- `data/_validation_report.json`
- `data/_validation_report.csv`

The script will optionally ask whether invalid `time_ms` rows should be removed.

In [ ]:
subprocess.run([sys.executable, str(ROOT / "tool" / "validate_downloaded_data.py")], check=False)

## 4) Time Coverage Visualization

Visualize UTC time coverage for every CSV in `data/`.
Each row is one CSV file. The x-axis is the global timeline across all files.
If a file covers the full timeline with no internal gaps, its horizontal line spans the full width.

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd()
if not (ROOT / "tool").exists() and (ROOT.parent / "tool").exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from tool.coverage_visualizer import CoverageConfig, CoverageVisualizer

viz_cfg = CoverageConfig(data_dir=ROOT / "data")
viz = CoverageVisualizer(viz_cfg)
viz.plot()

## 5) Verbal Analysis: Top 5 Shortest Time Horizons

Compute the time horizon of each CSV and print a concise textual analysis for the 5 shortest files.

In [ ]:
import pandas as pd

DATA_DIR = ROOT / "data"
csv_files = sorted([p for p in DATA_DIR.glob("*.csv") if not p.name.startswith("_")])

stats = []
for fp in csv_files:
    try:
        df = pd.read_csv(fp, usecols=["time_ms"])
    except Exception:
        continue

    ts = pd.to_numeric(df["time_ms"], errors="coerce").dropna().astype("int64")
    ts = ts[ts > 0]
    if ts.empty:
        continue

    t_min = int(ts.min())
    t_max = int(ts.max())
    horizon_ms = max(0, t_max - t_min)
    horizon_days = horizon_ms / (1000 * 60 * 60 * 24)

    stats.append({
        "file": fp.stem,
        "row_count": int(len(ts)),
        "start_utc": pd.to_datetime(t_min, unit="ms", utc=True),
        "end_utc": pd.to_datetime(t_max, unit="ms", utc=True),
        "horizon_days": horizon_days,
    })

if not stats:
    print("No valid CSV files with time_ms were found.")
else:
    top5 = sorted(stats, key=lambda x: x["horizon_days"])[:5]
    print("Top 5 shortest time horizons:\n")
    for i, item in enumerate(top5, start=1):
        print(
            f"{i}. {item['file']}\n"
            f"   - Rows: {item['row_count']}\n"
            f"   - Start (UTC): {item['start_utc']}\n"
            f"   - End   (UTC): {item['end_utc']}\n"
            f"   - Horizon: {item['horizon_days']:.2f} days\n"
        )

    avg_horizon = sum(x["horizon_days"] for x in top5) / len(top5)
    shortest = top5[0]
    longest_in_top5 = top5[-1]
    print("Summary:")
    print(f"- Average horizon among top 5 shortest files: {avg_horizon:.2f} days")
    print(f"- Shortest file: {shortest['file']} ({shortest['horizon_days']:.2f} days)")
    print(f"- 5th shortest file: {longest_in_top5['file']} ({longest_in_top5['horizon_days']:.2f} days)")

## 6) Data Alignment (Multi-Frequency)

Goal: build **4 aligned tables** (`1m`, `5m`, `15m`, `1h`) by joining all CSVs on `time_ms`.

What this section does:
1. Read every endpoint CSV in `data/`
2. Group files by frequency suffix (`_1m`, `_5m`, `_15m`, `_1h`)
3. Standardize each file (`time_ms` numeric, sorted, deduplicated)
4. Outer-join within each frequency on `time_ms` to get one aligned table per frequency
5. Add `time_utc`
6. Build a few starter features (safe defaults) for quant research

In [ ]:
from pathlib import Path
import re
import numpy as np
import pandas as pd

DATA_DIR = ROOT / "data"
ALIGNED_DIR = DATA_DIR / "aligned"
FREQS = ["1m", "5m", "15m", "1h"]
ALIGNED_DIR.mkdir(parents=True, exist_ok=True)


def detect_freq(stem: str):
    for f in FREQS:
        if stem.endswith(f"_{f}"):
            return f
    return None


def safe_file_prefix(stem: str, freq: str):
    # Remove trailing _<freq>, keep a compact prefix for feature names.
    return stem[: -(len(freq) + 1)] if stem.endswith(f"_{freq}") else stem


def load_one_csv(fp: Path):
    try:
        df = pd.read_csv(fp)
    except Exception:
        return None

    if "time_ms" not in df.columns:
        return None

    df["time_ms"] = pd.to_numeric(df["time_ms"], errors="coerce")
    df = df.dropna(subset=["time_ms"]).copy()
    if df.empty:
        return None

    df["time_ms"] = df["time_ms"].astype("int64")
    df = df.drop_duplicates(subset=["time_ms"]).sort_values("time_ms")
    return df


def add_starter_features(aligned_df: pd.DataFrame) -> pd.DataFrame:
    out = aligned_df.copy()

    # Try finding a likely price column to create returns/volatility features.
    price_candidates = [
        c for c in out.columns
        if c not in ("time_ms", "time_utc") and ("close" in c.lower() or "price" in c.lower())
    ]

    if price_candidates:
        px_col = price_candidates[0]
        out[px_col] = pd.to_numeric(out[px_col], errors="coerce")
        out["feat_ret_1"] = out[px_col].pct_change()
        out["feat_ret_5"] = out[px_col].pct_change(5)
        out["feat_vol_30"] = out["feat_ret_1"].rolling(30).std()

    # Row-wise missing ratio: useful to understand alignment quality.
    feature_cols = [c for c in out.columns if c not in ("time_ms", "time_utc")]
    if feature_cols:
        out["feat_missing_ratio"] = out[feature_cols].isna().mean(axis=1)

    return out


def drop_redundant_time_columns(df: pd.DataFrame) -> pd.DataFrame:
    # Keep only unified timeline fields: time_ms (index later) and time_utc.
    # Remove endpoint-specific time-like columns after alignment.
    time_like_suffixes = (
        "__time", "__timestamp", "__create_time", "__createTime", "__date", "__time_list"
    )
    drop_cols = [c for c in df.columns if c.endswith(time_like_suffixes)]
    if drop_cols:
        df = df.drop(columns=drop_cols, errors="ignore")
    return df


# 1) Discover and group CSVs by frequency
all_csvs = sorted([p for p in DATA_DIR.glob("*.csv") if not p.name.startswith("_")])
files_by_freq = {f: [] for f in FREQS}
for fp in all_csvs:
    freq = detect_freq(fp.stem)
    if freq:
        files_by_freq[freq].append(fp)

# 2) Build aligned table for each frequency
aligned_tables = {}

for freq in FREQS:
    fps = files_by_freq[freq]
    merged = None

    for fp in fps:
        df = load_one_csv(fp)
        if df is None:
            continue

        prefix = safe_file_prefix(fp.stem, freq)

        # Rename columns (except time_ms) to avoid collisions after merge.
        rename_map = {}
        for c in df.columns:
            if c == "time_ms":
                continue
            rename_map[c] = f"{prefix}__{c}"
        df = df.rename(columns=rename_map)

        if merged is None:
            merged = df
        else:
            merged = merged.merge(df, on="time_ms", how="outer")

    if merged is None:
        # Keep an empty indexed table for API consistency.
        aligned_tables[freq] = pd.DataFrame(columns=["time_utc"]).set_index(pd.Index([], name="time_ms"))
        continue

    merged = merged.sort_values("time_ms").reset_index(drop=True)
    merged["time_utc"] = pd.to_datetime(merged["time_ms"], unit="ms", utc=True)

    # Remove redundant endpoint-specific time columns after alignment.
    merged = drop_redundant_time_columns(merged)

    # 3) Feature engineering starter pack
    merged = add_starter_features(merged)

    # 4) Set time_ms as index for all aligned tables
    merged = merged.set_index("time_ms", drop=True)
    merged.index.name = "time_ms"

    aligned_tables[freq] = merged

    # 5) Save aligned table into data/aligned/
    out_csv = ALIGNED_DIR / f"aligned_{freq}.csv"
    merged.to_csv(out_csv)

# 6) Quick summary and samples
for freq in FREQS:
    t = aligned_tables[freq]
    print(f"\n[{freq}] aligned table shape: {t.shape}")
    if not t.empty:
        print("time range:", t["time_utc"].iloc[0], "->", t["time_utc"].iloc[-1])
        display(t.head(3))

print(f"\nAligned tables saved to: {ALIGNED_DIR}")

# Ready-to-use indexed tables:
# aligned_tables["1m"], aligned_tables["5m"], aligned_tables["15m"], aligned_tables["1h"]

In [ ]:
from pathlib import Path
import pandas as pd

FREQS = ["1m", "5m", "15m", "1h"]
ALIGNED_DIR = ROOT / "data" / "aligned"

# Prefer in-memory aligned tables from previous cells.
has_aligned_tables = "aligned_tables" in globals() and isinstance(aligned_tables, dict)

if has_aligned_tables:
    tables = {k: v for k, v in aligned_tables.items() if k in FREQS}
else:
    # Fallback: read from saved aligned CSV files.
    tables = {}
    for f in FREQS:
        p = ALIGNED_DIR / f"aligned_{f}.csv"
        if p.exists():
            df = pd.read_csv(p)
            if "time_ms" in df.columns:
                df = df.set_index("time_ms")
            tables[f] = df

if not tables:
    print("No aligned tables found. Please run the alignment section first.")
else:
    union_features = set()

    for f in FREQS:
        if f not in tables:
            continue
        df = tables[f]

        # Feature columns = all columns except explicit time display column.
        features = [c for c in df.columns if c != "time_utc"]
        union_features.update(features)

        print(f"\n[{f}] feature count: {len(features)}")
        for col in features:
            print(f"- {col}")

    print("\n============================================================")
    print(f"Union feature count across available aligned tables: {len(union_features)}")
    print("All unique features:")
    for col in sorted(union_features):
        print(f"- {col}")

## 7) 1m Aligned Table Example 

**Presented via self-developed TradingViewGrapher**

This section uses the 1m aligned table and automatically finds the first timestamp where these core features are all available:
- Close price
- EMA
- MA
- Bollinger bands (LB/MB/UB)
- MACD (MACD/Signal/Histogram)
- RSI

Then it plots two graphs (one for long term, one for short term)
* Default setting, longer term = 2500 oberservations; short term = 180 oberservations

In [ ]:
from pathlib import Path
import sys
import pandas as pd

# Resolve project root and ensure it is importable.
ROOT = Path.cwd()
if not (ROOT / "tool").exists() and (ROOT.parent / "tool").exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from tool.tradingview_graph import TradingViewGrapher

# Use in-memory aligned table if available; otherwise load from disk.
if "aligned_tables" in globals() and isinstance(aligned_tables, dict) and "1m" in aligned_tables:
    df_1m = aligned_tables["1m"].copy()
else:
    p = ROOT / "data" / "aligned" / "aligned_1m.csv"
    df_1m = pd.read_csv(p)
    if "time_ms" in df_1m.columns:
        df_1m = df_1m.set_index("time_ms")

plotter = TradingViewGrapher(max_points_full=2500, recent_minutes=180)
plotter.run(df_1m)

## 8) Cleaned Table (No Missing Features)

From this point, we are going to build the strategy using whale signals as top layers and CTA as core.

We are not going to use "whale index" and "top long short account" features in the following analysis.
Instead we will derive our own related features.

Build cleaned tables from aligned tables for each frequency (`1m`, `5m`, `15m`, `1h`):
1. Exclude columns whose names start with:
   - `futures_top_long_short_account_ratio_history_btcusdt_binance`
   - `futures_whale_index_history_btcusdt_binance`
2. Detect blackout gaps and keep only the most recent continuous interval
3. Locate the first timestamp where all remaining features are covered (within that interval)
4. Slice from that point onward
5. Remove any remaining rows with missing feature values
6. Save 4 cleaned tables to `data/cleaned/`

In [ ]:
from pathlib import Path
import sys
import importlib

ROOT = Path.cwd().resolve()
while not (ROOT / "tool").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import tool.cleaned_table as cleaned_table_module
cleaned_table_module = importlib.reload(cleaned_table_module)

# Rebuild cleaned tables from aligned tables with:
# 1) excluded whale/top-long-short feature families
# 2) exclude feat_missing_ratio / feat_ret* / feat_vol*
# 3) blackout-aware "most recent continuous interval" handling
#    (blackout is treated as long gap >= 1 day)
cleaned_tables = cleaned_table_module.build_cleaned_tables(
    root_dir=ROOT,
    freqs=("1m", "5m", "15m", "1h"),
    blackout_gap_ms=86_400_000,
)

# Outputs ready:
# cleaned_tables["1m"], cleaned_tables["5m"], cleaned_tables["15m"], cleaned_tables["1h"]

## 9) Rebuild `cleaned_1m` (OHLCV only) + apply patch + Merge with `whale_features_1m`

This part does three things:
1. Rebuild `data/cleaned/cleaned_1m.csv` from `data/aligned/aligned_1m.csv` using only OHLCV fields.
2. Apply optional OHLCV patch CSV (e.g. `data/price_1m_patch_20260305_okx_perp.csv`). If the filename contains `okx`: `volume` is **BTC** → multiply by **close** (USDT notional), then multiply by **`ratio`** from `data/backfill/okx_binance_volume_ratio.json` (from `OHLCV_1m_Backfill_Research.ipynb`, calibrated on UTC 2026-03-04). If the JSON is missing, only BTC×close is applied.
3. Merge rebuilt `cleaned_1m` with `data/whale_features_1m.csv` by minute-level UTC timestamp.

Alignment rule:
- `cleaned_1m.time_utc` <-> `whale_features_1m.minute_utc`
- both parsed as UTC and floored to minute.

In [ ]:
# A) Rebuild cleaned_1m from aligned_1m with OHLCV only (+ optional patch fill)
from pathlib import Path
import json
import pandas as pd

ROOT = Path.cwd()
if not (ROOT / "data").exists() and (ROOT.parent / "data").exists():
    ROOT = ROOT.parent

ALIGNED_1M = ROOT / "data" / "aligned" / "aligned_1m.csv"
CLEANED_1M = ROOT / "data" / "cleaned" / "cleaned_1m.csv"
PATCH_1M = ROOT / "data" / "price_1m_patch_20260305_okx_perp.csv"

price_cols = [
    "futures_price_history_btcusdt_binance__open",
    "futures_price_history_btcusdt_binance__high",
    "futures_price_history_btcusdt_binance__low",
    "futures_price_history_btcusdt_binance__close",
    "futures_price_history_btcusdt_binance__volume_usd",
]

raw = pd.read_csv(ALIGNED_1M, usecols=["time_ms", "time_utc"] + price_cols)
raw["time_ms"] = pd.to_numeric(raw["time_ms"], errors="coerce")
raw["time_utc"] = pd.to_datetime(raw["time_utc"], utc=True, errors="coerce")
for c in price_cols:
    raw[c] = pd.to_numeric(raw[c], errors="coerce")

# Apply patch if available (fills OHLCV null gap)
patch_applied_rows = 0
if PATCH_1M.exists():
    patch = pd.read_csv(PATCH_1M)
    patch["time_ms"] = pd.to_numeric(patch["time_ms"], errors="coerce")
    patch["time_utc"] = pd.to_datetime(patch["time_utc"], utc=True, errors="coerce")
    for c in ["open", "high", "low", "close", "volume"]:
        patch[c] = pd.to_numeric(patch[c], errors="coerce")

    # OKX: ccxt volume = BTC; notional ≈ vol_btc * close; then scale to Binance level via OHLCV notebook JSON
    if "okx" in PATCH_1M.name.lower():
        patch["volume"] = patch["volume"] * patch["close"]
        _ratio_json = ROOT / "data" / "backfill" / "okx_binance_volume_ratio.json"
        if _ratio_json.exists():
            _r = float(json.loads(_ratio_json.read_text(encoding="utf-8"))["ratio"])
            patch["volume"] = patch["volume"] * _r

    patch = patch.rename(
        columns={
            "open": "futures_price_history_btcusdt_binance__open",
            "high": "futures_price_history_btcusdt_binance__high",
            "low": "futures_price_history_btcusdt_binance__low",
            "close": "futures_price_history_btcusdt_binance__close",
            "volume": "futures_price_history_btcusdt_binance__volume_usd",
        }
    )

    patch_fill = patch[["time_ms"] + price_cols].drop_duplicates(subset=["time_ms"], keep="first")
    raw = raw.merge(patch_fill, on="time_ms", how="left", suffixes=("", "__patch"))

    for c in price_cols:
        alt = c + "__patch"
        before_na = raw[c].isna().sum()
        raw[c] = raw[c].combine_first(raw[alt])
        after_na = raw[c].isna().sum()
        patch_applied_rows += int(before_na - after_na)
        raw = raw.drop(columns=[alt])

cleaned_1m = (
    raw.dropna(subset=["time_ms", "time_utc"] + price_cols)
    .sort_values("time_ms")
    .drop_duplicates(subset=["time_ms"], keep="first")
    .rename(
        columns={
            "futures_price_history_btcusdt_binance__open": "open",
            "futures_price_history_btcusdt_binance__high": "high",
            "futures_price_history_btcusdt_binance__low": "low",
            "futures_price_history_btcusdt_binance__close": "close",
            "futures_price_history_btcusdt_binance__volume_usd": "volume",
        }
    )
)

cleaned_1m.to_csv(CLEANED_1M, index=False)
print("Saved:", CLEANED_1M)
print("shape:", cleaned_1m.shape)
print("start:", cleaned_1m["time_utc"].iloc[0], "end:", cleaned_1m["time_utc"].iloc[-1])
print("Patch file used:", PATCH_1M.exists(), "patched_value_count:", patch_applied_rows)
display(cleaned_1m.head(3))

In [ ]:
# B) Merge cleaned_1m and whale_features_1m by minute UTC
WHALE_1M = ROOT / "data" / "whale_features_1m.csv"
MERGED_1M = ROOT / "data" / "cleaned" / "cleaned_1m_with_whale.csv"

price = pd.read_csv(CLEANED_1M)
whale = pd.read_csv(WHALE_1M)

# avoid ambiguous duplicated time_ms after merge
if "time_ms" in price.columns:
    price = price.rename(columns={"time_ms": "price_time_ms"})
if "time_ms" in whale.columns:
    whale = whale.rename(columns={"time_ms": "whale_time_ms"})

price["time_utc"] = pd.to_datetime(price["time_utc"], utc=True, errors="coerce").dt.floor("min")
whale["minute_utc"] = pd.to_datetime(whale["minute_utc"], utc=True, errors="coerce").dt.floor("min")

# Use all non-time columns from both tables
price_non_time = [c for c in price.columns if c not in {"time_utc"}]
whale_non_time = [c for c in whale.columns if c not in {"minute_utc"}]

price_use = price[["time_utc"] + price_non_time].copy()
whale_use = whale[["minute_utc"] + whale_non_time].copy()

merged = price_use.merge(
    whale_use,
    left_on="time_utc",
    right_on="minute_utc",
    how="left",
)

# keep time keys in front for readability
cols_front = [c for c in ["time_utc", "price_time_ms", "minute_utc", "whale_time_ms"] if c in merged.columns]
other_cols = [c for c in merged.columns if c not in cols_front]
merged = merged[cols_front + other_cols]

merged.to_csv(MERGED_1M, index=False)
print("Saved:", MERGED_1M)
print("shape:", merged.shape)

coverage = merged["minute_utc"].notna().mean() * 100 if "minute_utc" in merged.columns else float("nan")
print(f"Whale row coverage on cleaned_1m timeline: {coverage:.2f}%")
print("Merged range:", merged["time_utc"].min(), "->", merged["time_utc"].max())
display(merged.tail(3))

## 10) Validate all `whale_features_1m` columns are fully merged

This section validates whether **all non-time columns** from `data/whale_features_1m.csv` have been carried into `data/cleaned/cleaned_1m_with_whale.csv`.

Validation checklist:
1. Schema completeness: all whale non-time columns exist in merged table.
2. Timestamp coverage: how many whale minutes are present on merged timeline.
3. Value consistency: random matched rows are checked against original whale table.

In [ ]:
# Validate whale-feature merge completeness
import numpy as np
import pandas as pd

WHALE_1M = ROOT / "data" / "whale_features_1m.csv"
MERGED_1M = ROOT / "data" / "cleaned" / "cleaned_1m_with_whale.csv"

whale = pd.read_csv(WHALE_1M)
merged = pd.read_csv(MERGED_1M)

whale["minute_utc"] = pd.to_datetime(whale["minute_utc"], utc=True, errors="coerce").dt.floor("min")
merged["time_utc"] = pd.to_datetime(merged["time_utc"], utc=True, errors="coerce").dt.floor("min")
if "minute_utc" in merged.columns:
    merged["minute_utc"] = pd.to_datetime(merged["minute_utc"], utc=True, errors="coerce").dt.floor("min")

# 1) Schema completeness
whale_non_time_cols = [c for c in whale.columns if c not in {"minute_utc", "time_ms"}]
missing_in_merged = sorted(set(whale_non_time_cols) - set(merged.columns))

# 2) Timestamp coverage (whale timeline -> merged timeline)
whale_minutes = set(whale["minute_utc"].dropna().unique().tolist())
merged_minutes = set(merged["time_utc"].dropna().unique().tolist())
whale_minutes_in_merged = whale_minutes.intersection(merged_minutes)

# 3) Value consistency check on matched minutes
#    (for rows where whale info is available in merged)
matched = merged.dropna(subset=["minute_utc"]).copy() if "minute_utc" in merged.columns else pd.DataFrame()
consistency_rows = []
if not matched.empty and not missing_in_merged:
    whale_idx = whale.drop_duplicates(subset=["minute_utc"], keep="first").set_index("minute_utc")
    probe_cols = whale_non_time_cols[: min(12, len(whale_non_time_cols))]  # sample first 12 columns for quick check
    sample_n = min(200, len(matched))
    sample = matched.sample(sample_n, random_state=42)

    for c in probe_cols:
        left = sample[c]
        right = whale_idx.reindex(sample["minute_utc"])[c].values

        # numeric columns: tolerance-based; non-numeric: exact match
        left_num = pd.to_numeric(left, errors="coerce")
        right_num = pd.to_numeric(pd.Series(right), errors="coerce")
        if left_num.notna().sum() > 0 or right_num.notna().sum() > 0:
            same = np.isclose(left_num.fillna(np.nan), right_num.fillna(np.nan), equal_nan=True, rtol=1e-9, atol=1e-9)
            ratio = float(np.mean(same))
        else:
            same = (left.astype(str).fillna("<NA>").to_numpy() == pd.Series(right).astype(str).fillna("<NA>").to_numpy())
            ratio = float(np.mean(same))

        consistency_rows.append({"column": c, "match_ratio_on_sample": ratio})

consistency_df = pd.DataFrame(consistency_rows).sort_values("match_ratio_on_sample") if consistency_rows else pd.DataFrame()

status = pd.DataFrame(
    [
        {
            "check": "all whale non-time columns exist in merged",
            "pass": len(missing_in_merged) == 0,
            "detail": f"missing_count={len(missing_in_merged)}",
        },
        {
            "check": "whale minute timestamps covered by merged timeline",
            "pass": len(whale_minutes) == len(whale_minutes_in_merged),
            "detail": f"covered={len(whale_minutes_in_merged)}/{len(whale_minutes)}",
        },
        {
            "check": "sample value consistency (selected columns)",
            "pass": consistency_df.empty or float(consistency_df["match_ratio_on_sample"].min()) >= 0.999,
            "detail": "min_match_ratio=" + (f"{float(consistency_df['match_ratio_on_sample'].min()):.4f}" if not consistency_df.empty else "N/A"),
        },
    ]
)

display(status)

print("\nMissing whale columns in merged (if any):")
print(missing_in_merged[:30], "..." if len(missing_in_merged) > 30 else "")

print("\nCoverage summary:")
print(f"Whale minutes total: {len(whale_minutes)}")
print(f"Whale minutes in merged timeline: {len(whale_minutes_in_merged)}")

if not consistency_df.empty:
    print("\nSample consistency (first 12 whale columns):")
    display(consistency_df)
else:
    print("\nNo consistency sample generated (likely no matched rows).")

## 11) Demo view of `cleaned_1m_with_whale` for PPT (high-vol window near 2026-03-26 00:00 UTC)

This section extracts a minute-level demo table from `cleaned_1m_with_whale` around a strong move near midnight on **2026-03-26 (UTC)**.

Goal:
- Show a compact, interpretable sample table for slides
- Include both price/volume and whale columns

In [ ]:
# Load merged minute table
MERGED_1M = ROOT / "data" / "cleaned" / "cleaned_1m_with_whale.csv"
show_df = pd.read_csv(MERGED_1M)
show_df["time_utc"] = pd.to_datetime(show_df["time_utc"], utc=True, errors="coerce")

for c in ["open", "high", "low", "close", "volume", "whale_alert_count", "whale_alert_notional_abs", "whale_pos_notional_abs", "whale_pos_lev_wavg"]:
    if c in show_df.columns:
        show_df[c] = pd.to_numeric(show_df[c], errors="coerce")

show_df = show_df.sort_values("time_utc").reset_index(drop=True)
show_df["ret_1m"] = show_df["close"].pct_change()
show_df["abs_ret_1m_bp"] = show_df["ret_1m"].abs() * 1e4

# Focus around midnight on 2026-03-26 UTC
focus_start = pd.Timestamp("2026-03-25 22:00:00", tz="UTC")
focus_end = pd.Timestamp("2026-03-26 03:00:00", tz="UTC")
focus = show_df[(show_df["time_utc"] >= focus_start) & (show_df["time_utc"] <= focus_end)].copy()

if focus.empty:
    raise ValueError("No rows found in focus range.")

# Find strongest 1m move in this window and take +/- 20 minutes
center_idx = focus["abs_ret_1m_bp"].idxmax()
center_ts = show_df.loc[center_idx, "time_utc"]
win_start = center_ts - pd.Timedelta(minutes=20)
win_end = center_ts + pd.Timedelta(minutes=20)

demo = show_df[(show_df["time_utc"] >= win_start) & (show_df["time_utc"] <= win_end)].copy()

# Prepare PPT-friendly view
demo_cols = [
    "time_utc", "open", "high", "low", "close", "volume",
    "ret_1m", "abs_ret_1m_bp",
    "whale_alert_count", "whale_alert_notional_abs",
    "whale_pos_notional_abs", "whale_pos_lev_wavg",
]
demo_cols = [c for c in demo_cols if c in demo.columns]

demo_show = demo[demo_cols].copy()
demo_show["ret_1m"] = demo_show["ret_1m"].round(6)
demo_show["abs_ret_1m_bp"] = demo_show["abs_ret_1m_bp"].round(2)
for c in ["open", "high", "low", "close", "volume", "whale_alert_count", "whale_alert_notional_abs", "whale_pos_notional_abs", "whale_pos_lev_wavg"]:
    if c in demo_show.columns:
        demo_show[c] = demo_show[c].round(4)

print("Center shock time (max |1m ret| near midnight):", center_ts)
print("Demo window:", win_start, "->", win_end, "rows:", len(demo_show))
print("Nonzero whale_alert_count ratio in demo window:", f"{((demo_show['whale_alert_count'].fillna(0) > 0).mean() * 100):.2f}%" if 'whale_alert_count' in demo_show.columns else "N/A")

display(demo_show)

# Optional: save standalone demo table for PPT
DEMO_OUT = ROOT / "data" / "cleaned" / "cleaned_1m_with_whale_demo_20260326_midnight.csv"
demo_show.to_csv(DEMO_OUT, index=False)
print("Saved demo table:", DEMO_OUT)

### 11B) PPT compact demo: keep only alert-active rows in the same window

Because whale alerts are sparse at minute level, this view keeps only rows where alert is active (`whale_alert_count > 0`) within the selected demo window.

In [ ]:
# 11B) Keep only alert-active rows from the demo window
# Reuse demo_show if it exists; otherwise build from demo
if "demo_show" in globals():
    base_demo = demo_show.copy()
elif "demo" in globals():
    base_demo = demo.copy()
else:
    raise ValueError("Run Part 11 first to create demo window data.")

if "whale_alert_count" not in base_demo.columns:
    raise ValueError("Column whale_alert_count not found in demo table.")

compact = base_demo[base_demo["whale_alert_count"].fillna(0) > 0].copy()

# Sort by alert intensity then time for PPT readability
compact = compact.sort_values(["whale_alert_count", "time_utc"], ascending=[False, True])

print("Alert-active rows in demo window:", len(compact))
display(compact)

COMPACT_OUT = ROOT / "data" / "cleaned" / "cleaned_1m_with_whale_demo_20260326_midnight_alert_only.csv"
compact.to_csv(COMPACT_OUT, index=False)
print("Saved compact demo table:", COMPACT_OUT)

### 11C) March 26 alert-active Top 10 rows (for PPT)

This compact table uses **2026-03-26 (UTC)** only and keeps the top 10 alert-active rows by:
1. `whale_alert_count` (desc)
2. `whale_alert_notional_abs` (desc)

In [ ]:
# 11C) March 26 top-10 nonzero alert rows
MARCH26_START = pd.Timestamp("2026-03-26 00:00:00", tz="UTC")
MARCH26_END = pd.Timestamp("2026-03-26 23:59:59", tz="UTC")

ppt_src = pd.read_csv(ROOT / "data" / "cleaned" / "cleaned_1m_with_whale.csv")
ppt_src["time_utc"] = pd.to_datetime(ppt_src["time_utc"], utc=True, errors="coerce")

for c in ["open", "high", "low", "close", "volume", "whale_alert_count", "whale_alert_notional_abs", "whale_pos_notional_abs", "whale_pos_lev_wavg"]:
    if c in ppt_src.columns:
        ppt_src[c] = pd.to_numeric(ppt_src[c], errors="coerce")

march26 = ppt_src[(ppt_src["time_utc"] >= MARCH26_START) & (ppt_src["time_utc"] <= MARCH26_END)].copy()
march26_nz = march26[march26["whale_alert_count"].fillna(0) > 0].copy()

cols = [
    "time_utc", "open", "high", "low", "close", "volume",
    "whale_alert_count", "whale_alert_notional_abs",
    "whale_pos_notional_abs", "whale_pos_lev_wavg",
]
cols = [c for c in cols if c in march26_nz.columns]

march26_top10 = (
    march26_nz[cols]
    .sort_values(["whale_alert_count", "whale_alert_notional_abs"], ascending=[False, False])
    .head(10)
    .sort_values("time_utc", ascending=True)
    .copy()
)

print("March 26 rows:", len(march26), "| nonzero alert rows:", len(march26_nz))
display(march26_top10)

OUT_TOP10 = ROOT / "data" / "cleaned" / "cleaned_1m_with_whale_demo_20260326_alert_top10.csv"
march26_top10.to_csv(OUT_TOP10, index=False)
print("Saved:", OUT_TOP10)